# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id`, ensuring reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare the mlcroissant Dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
metadata = dataset.metadata
print("Dataset Name:", getattr(metadata, 'name', None))
print("Dataset Description:\n", getattr(metadata, 'description', None))
print("Date Published:", getattr(metadata, 'datePublished', None))
print("License:", getattr(metadata, 'license', None))


## 2. Data Overview
Review available record sets, fields, and their IDs. We reference all entities by their `@id`.

In [ ]:
# List all record sets available in the dataset, by their @id
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # Try extracting from the main metadata dict (for older schemas)
    try:
        # This fallback attempts direct attribute or dict access
        record_sets = metadata['recordSet']
    except Exception:
        print("No record sets available in the dataset metadata.")

print(f"Number of record sets: {len(record_sets)}\n")

for idx, recset in enumerate(record_sets):
    recset_id = getattr(recset, '@id', None) if hasattr(recset, '@id') else recset.get('@id', None)
    print(f"RecordSet {idx+1}: @id = {recset_id}")
    fields = getattr(recset, 'field', [])
    # Print fields and columns for the record set
    print("  Fields:")
    for field in fields:
        field_id = getattr(field, '@id', None) if hasattr(field, '@id') else field.get('@id', None)
        print(f"    - field @id: {field_id}")
        # If field maps to a column, print
        columns = getattr(field, 'column', [])
        if not isinstance(columns, list):
            columns = [columns]
        for col in columns:
            col_id = getattr(col, '@id', None) if hasattr(col, '@id') else (col.get('@id', None) if isinstance(col, dict) else col)
            if col_id:
                print(f"      - column @id: {col_id}")
    print("\n")

# For demonstration, if record_sets is empty (as here), try loading via dataset.record_set_ids
if not record_sets:
    record_set_ids = [x for x in dataset.record_set_ids]
    print(f"Loaded record set IDs from dataset: {record_set_ids}")
else:
    record_set_ids = [getattr(recset, '@id', None) if hasattr(recset, '@id') else recset.get('@id', None) for recset in record_sets]


## 3. Data Extraction
Load data from each record set into a pandas DataFrame. All operations use `@id` values for the record set and fields.

In [ ]:
from mlcroissant.types import CroissantException
dataframes = {}
record_set_ids = record_set_ids if 'record_set_ids' in globals() else list(dataset.record_set_ids)

print("Extracting data from record sets...\n")
for recset_id in record_set_ids:
    print(f"- Extracting records from RecordSet @id: {recset_id}")
    try:
        df = pd.DataFrame(dataset.records(record_set=recset_id))
        dataframes[recset_id] = df
        print(f"    Columns: {df.columns.tolist()}")
        print(df.head(2), "\n")
    except CroissantException as e:
        print(f"    Could not load record set '{recset_id}': {str(e)}.")
    except Exception as e:
        print(f"    Unexpected error for '{recset_id}': {e}\n")

# For following exploration, select the main record set
if dataframes:
    # Pick the largest DataFrame as main
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Selected RecordSet for analysis: {main_record_set_id}")
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping. All fields/columns are referenced by their `@id`. *Choose a numeric field and a group field after inspecting the DataFrame columns above.*

In [ ]:
# For demonstration, pick known field @ids for this proteomics dataset
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    # You can inspect columns to pick suitable fields
    print("Main DataFrame columns (potential field @ids):")
    print(main_df.columns.tolist())

    # Try to pick a numeric field, e.g., 'cr:normalized_abundance', common in proteomics tables
    # Fallback: pick the first float/integer column
    for col in main_df.columns:
        # Try to find a numerical column
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

    # Try a typical group field e.g. 'cr:accession' or 'cr:protein_group'
    possible_group_fields = ['cr:accession', 'cr:protein_group', 'cr:description','cr:sample']
    for col in main_df.columns:
        if col in possible_group_fields:
            group_field_id = col
            break
    if not group_field_id and len(main_df.columns) > 0:
        group_field_id = main_df.columns[0]

    print(f"\nChosen numeric field (@id): {numeric_field_id}\nChosen group field (@id): {group_field_id}\n")

    # EDA: Filtering
    threshold = main_df[numeric_field_id].mean() if numeric_field_id else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold] if numeric_field_id else main_df.copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    if numeric_field_id:
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())


## 5. Visualization
Visualize the distribution of a numeric field using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id], bins=35, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped data available, plot group averages
    if group_field_id and group_field_id in main_df.columns:
        group_means = main_df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(10, 5))
        group_means.sort_values(ascending=False).head(20).plot(kind='bar')
        plt.title(f"Top 20 groups by mean {numeric_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:

- Load a Croissant-described mass spectrometry proteomics dataset using `mlcroissant`
- Explore all available record sets, fields, and reference their `@id`
- Extract data dynamically using the provided schema
- Run basic EDA: filtering, normalization, grouping, and visualization

This approach ensures transparency and reproducibility in FAIR data workflows. For more advanced analysis, further domain-specific processing and hypothesis testing can be layered on top of this foundation.